# Preview Visual das Rotas com Romaneio Real e Validação da API Pedestre

Este notebook permite:
- Visualizar rotas otimizadas com base no romaneio real
- Validar a API de roteirização pedestre (OSRM/OpenRouteService)
- Analisar o balanceamento e a qualidade das rotas para entregadores a pé na Zona Sul do Rio de Janeiro

In [ ]:
# 1. Importar Bibliotecas Necessárias
import pandas as pd
import folium
import requests
import json
from shapely.geometry import shape, Point
import geopandas as gpd
from folium.plugins import MarkerCluster
from IPython.display import display

In [ ]:
# 2. Carregar Dados das Rotas e Romaneios
# Carregar romaneio real
romaneio = pd.read_csv('../romaneio.txt', sep='\t')
romaneio['Latitude'] = romaneio['Latitude'].str.replace(',', '.').astype(float)
romaneio['Longitude'] = romaneio['Longitude'].str.replace(',', '.').astype(float)

# Carregar GeoJSON dos bairros da Zona Sul
with open('../data/geojson/zona_sul_rio.json', 'r', encoding='utf-8') as f:
    geojson_zs = json.load(f)

# Converter para GeoDataFrame
bairros_gdf = gpd.GeoDataFrame.from_features(geojson_zs['features'])

# Exibir amostra dos dados
display(romaneio.head())
display(bairros_gdf[['name', 'geometry']].head())

In [ ]:
# 3. Visualizar Rotas em Mapa Interativo
# Mapa centrado no Leblon
center = [romaneio['Latitude'].mean(), romaneio['Longitude'].mean()]
m = folium.Map(location=center, zoom_start=15, tiles='cartodbpositron')

# Adicionar bairros (GeoJSON)
folium.GeoJson(geojson_zs, name='Bairros', style_function=lambda f: {
    'fillColor': f['properties'].get('color', '#cccccc'),
    'color': '#333', 'weight': 1, 'fillOpacity': 0.15
}).add_to(m)

# Adicionar pontos do romaneio
mc = MarkerCluster(name='Pacotes').add_to(m)
for idx, row in romaneio.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"NF: {row['NF']}<br>{row['Endereço Completo']}",
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(mc)

m

In [ ]:
# 4. Comparar Rotas Previstas com Romaneio Real
# (Simulação: agrupar por latitude/longitude próximas para simular clusters)
from sklearn.cluster import KMeans
import numpy as np

coords = romaneio[['Latitude', 'Longitude']].values
k = 2  # Exemplo: 2 entregadores
kmeans = KMeans(n_clusters=k, random_state=42).fit(coords)
romaneio['cluster'] = kmeans.labels_

# Visualizar clusters no mapa
colors = ['red', 'green', 'blue', 'purple', 'orange']
for i in range(k):
    cluster_points = romaneio[romaneio['cluster'] == i]
    for idx, row in cluster_points.iterrows():
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=6,
            color=colors[i % len(colors)],
            fill=True,
            fill_opacity=0.7,
            popup=f"Cluster {i} - NF: {row['NF']}"
        ).add_to(m)
m

In [ ]:
# 5. Validar Respostas da API Pedestre
# Exemplo: enviar rota de cada cluster para API OSRM/OpenRouteService
api_url = "http://localhost:5000/route/v1/foot/"

results = []
for i in range(k):
    cluster_points = romaneio[romaneio['cluster'] == i]
    coords = cluster_points[['Longitude', 'Latitude']].values
    coords_str = ";".join([f"{lng},{lat}" for lng, lat in coords])
    url = f"{api_url}{coords_str}?overview=full&geometries=geojson"
    try:
        resp = requests.get(url)
        if resp.ok:
            data = resp.json()
            dist_km = data['routes'][0]['distance'] / 1000
            duration_min = data['routes'][0]['duration'] / 60
            results.append({
                'cluster': i,
                'dist_km': dist_km,
                'duration_min': duration_min,
                'status': 'ok'
            })
        else:
            results.append({'cluster': i, 'status': 'fail', 'error': resp.text})
    except Exception as e:
        results.append({'cluster': i, 'status': 'fail', 'error': str(e)})

import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)

# 6. Exibir Resultados de Validação

- A tabela acima mostra a distância e o tempo estimado de cada cluster/rota segundo a API pedestre.
- Se houver falha, verifique se a API OSRM/OpenRouteService está rodando e acessível.
- O mapa acima permite validar visualmente a distribuição dos pacotes e clusters.